In [1]:
from pipeline_wlog import Pipeline
from utils.logging_setup import setup_latency_logger
from utils.audio_streaming import stream_audio
from utils.audio_preprocessing import preprocess_audio
from threading import Thread
import sounddevice as sd
import time
import os

In [2]:
def run_pipeline(wav_path, input_language="en", output_language="da", min_chunk_size=1000):
    audio = preprocess_audio(wav_path)

    base_name = os.path.splitext(os.path.basename(wav_path))[0]
    # build a filename to distinguish different chunk sizes
    file_name = f"{base_name}_chunk{min_chunk_size}"

    pipeline = Pipeline(input_language=input_language, output_language=output_language, min_chunk_size=min_chunk_size, file_name=file_name, device="mps")
    pipeline.start()

    def print_outputs(queue):
        while True:
            result = queue.get()
            if result is None:
                break
            transcript, translated, audio = result
            sd.play(audio, 16000)
            sd.wait()

    printer_thread = Thread(target=print_outputs, args=(pipeline.output_queue,))
    printer_thread.start()

    for i, chunk in enumerate(stream_audio(audio, frame_ms=200)):
        start_time = time.perf_counter()

        # Log the time when the audio chunk is sent to the pipeline (each sample is enumerated and time is logged) (!OBS: this is samples, not chunks)
        # But the time is only logged for the last sample of each chunk, so it will not log all samples.
        pipeline.audio_queue.put((i, start_time, chunk))

        
    pipeline.stop()
    printer_thread.join()

In [4]:
dk_data = ["data/danish/dk_speaker_1.wav", 
           "data/danish/dk_speaker_2.wav", 
           "data/danish/dk_speaker_3.wav", 
           "data/danish/dk_speaker_4.wav", 
           "data/danish/dk_speaker_5.wav",
           "data/danish/dk_speaker_6.wav",
           "data/danish/dk_speaker_7.wav",
           "data/danish/dk_speaker_8.wav",
           "data/danish/dk_speaker_9.wav",
           "data/danish/dk_speaker_10.wav"]

en_data = ["data/english/speaker_1_final.wav",
           "data/english/speaker_2_final.wav",
           "data/english/speaker_3_final.wav",
           "data/english/speaker_4_final.wav",
           "data/english/speaker_5_final.wav",
           "data/english/speaker_6_final.wav",
           "data/english/speaker_7_final.wav",
           "data/english/speaker_8_final.wav",
           "data/english/speaker_9_final.wav",
           "data/english/speaker_10_final.wav"]

chunk_sizes_testing = [3000, 3500, 4000, 4500, 5000, 5500, 6000]

--- 

## Danish to English

In [8]:
input_language = "da"
output_language = "en"

for chunk_size in chunk_sizes_testing:
    for input_file in dk_data:
        base_name = os.path.splitext(os.path.basename(input_file))[0]
        # build a filename to distinguish different chunk sizes
        file_name = f"{base_name}_chunk{chunk_size}"

        setup_latency_logger(file_name=file_name)
        results = run_pipeline(input_file, input_language=input_language, output_language=output_language, min_chunk_size=chunk_size)
        print("logging complete")

Loading Silero-VAD …
Loading SpeechT5 model: microsoft/speecht5_tts
Silero-VAD initialised!
Loading Danish transcriber model…


/Users/claralouisebrodt/anaconda3/envs/02466_AI_dubbing/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:177: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 67378.38it/s]


SpeechT5 model loaded!


Device set to use mps:0


Danish transcriber model loaded


...

SpeechT5 model loaded!


Device set to use mps:0


Danish transcriber model loaded


2025-06-15 20:48:13,241 - 32,434.51,945.10,8876.69,39781.74, en dag gik los angeles times kolonisten steve, One day the hour of the Angeles colonized steve.
2025-06-15 20:48:23,918 - 58,277.77,1166.23,10675.10,45072.38, lopes gennem gaderne i centrum af los angeles da han hørte smuk musik, loopes through the streets in the center of los angeles when he heard beautiful music
2025-06-15 20:48:29,203 - 87,351.86,1196.34,5284.63,44310.21, kilden var en charmerende ro afrikansk amerikansk hjemløs mand der spillede på, the source was a charming calm African American homeless man playing at
2025-06-15 20:48:34,291 - 112,350.27,1169.90,5087.18,44204.10, de spillede på en violin der kun havde strenge jeg fortæller en historie som mange, they played on a violin that only had strings I tell a story like many
2025-06-15 20:48:38,722 - 137,153.48,1231.06,4430.75,43459.11, så mange af kender fordi steves kolonner blev grundlaget for en bog der blev, so many of know because steve's columns became the

logging complete


---

## English to Danish

In [12]:
input_language = "en"
output_language = "da"

for chunk_size in chunk_sizes_testing:
    for input_file in en_data:
        base_name = os.path.splitext(os.path.basename(input_file))[0]
        # build a filename to distinguish different chunk sizes
        file_name = f"{base_name}_chunk{chunk_size}"

        setup_latency_logger(file_name=file_name)
        results = run_pipeline(input_file, input_language=input_language, output_language=output_language, min_chunk_size=chunk_size)
        print("logging complete")

Loading Silero-VAD …
Loading SpeechT5 Danish model: JackismyShephard/speecht5_tts-finetuned-nst-da
Silero-VAD initialised!
Loading Faster Whisper model: base.en …


/Users/claralouisebrodt/anaconda3/envs/02466_AI_dubbing/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:177: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Whisper model loaded!
Danish SpeechT5 model loaded!


2025-06-16 17:50:51,092 - 15,638.95,770.81,19503.35,27317.51, I'd like to share with you a discovery that I made., Jeg vil gerne dele en opdagelse med dig.
2025-06-16 17:50:59,601 - 33,629.30,1331.01,8506.92,32159.37, A few months ago while writing an article for the Italian one..., et par måneder siden mens du skriver en artikel til den italiensk én...
2025-06-16 17:51:02,584 - 48,585.22,1065.70,2982.81,32073.88, I always keep my bazaar as handy., jeg altid holde min basar så handy.
2025-06-16 17:51:09,475 - 63,663.36,1177.26,6891.27,35904.86, I don't know if I'm writing anything but I had already finished editing the bazaar., jeg ved ikke om jeg skriver noget men jeg havde allerede færdig redigere basaren.
2025-06-16 17:51:13,663 - 79,659.98,1423.59,4187.45,36823.81, And I realized that I had never once in my life., og jeg indså at jeg aldrig havde haft en eneste gang i mit liv.
2025-06-16 17:51:17,873 - 96,538.74,1118.34,4209.43,37570.90, Looked up the word disabled to see what I'd 

logging complete
Loading Silero-VAD …
Loading SpeechT5 Danish model: JackismyShephard/speecht5_tts-finetuned-nst-da
Silero-VAD initialised!
Loading Faster Whisper model: base.en …


...

logging complete
Loading Silero-VAD …
Loading SpeechT5 Danish model: JackismyShephard/speecht5_tts-finetuned-nst-da
Silero-VAD initialised!
Loading Faster Whisper model: base.en …
Whisper model loaded!


/Users/claralouisebrodt/anaconda3/envs/02466_AI_dubbing/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:177: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Danish SpeechT5 model loaded!


2025-06-16 20:56:31,438 - 32,649.81,2029.83,21264.17,24873.30, One day Los Angeles Times columnist Steve Lopez was walking along the streets of downtown Los Angeles when he heard, En dag gik Los Angeles som var klummeskribent Steve Lopez langs gaderne i Los Angeles da han hørte
2025-06-16 20:56:36,385 - 65,591.16,1838.12,4946.75,22978.35, beautiful music. And the source was a man an African-American man., smuk musik. og kilden var en mand en afrikansk-amerikansk mand.
2025-06-16 20:56:41,343 - 96,567.90,1348.62,4958.08,21475.47, Charming rugged homeless playing a violin that only had two strings., charmerende robust hjemløs spille en violin der kun havde to strenge.
2025-06-16 20:56:47,245 - 127,633.23,1516.38,5901.53,20967.72, I'm telling a story that many of you know because Steve's columns became the basis for a book., jeg fortæller en historie som mange af jer kender fordi Steves kolonner blev grundlaget for en bog.
2025-06-16 20:56:54,803 - 157,649.52,2001.43,7556.87,22337.37, Whi

logging complete
